# Build APK Android cho PianoGame bằng Google Colab

Notebook này cài môi trường Android/Rust/Node trên Colab rồi build APK Tauri Android cho dự án PianoGame.

> Lưu ý: lần chạy đầu vẫn phải tải Android SDK/NDK, Rust target, npm package và Gradle dependency nên chưa chắc nhanh ngay. Nếu giữ nguyên một Colab runtime rồi build lại nhiều lần thì thường nhanh hơn GitHub Actions.

## Không để lộ keystore

- Không commit file `.jks`/`.keystore` lên repo.
- Khi ký APK trong Colab, chỉ upload keystore vào runtime tạm thời của Colab.
- Mật khẩu nhập bằng `getpass`, notebook không in mật khẩu ra output.


## 1. Cấu hình build

Điền `REPO_URL` nếu muốn Colab tự clone repo. Nếu repo private, dùng URL có token hoặc tự upload source vào `/content/PianoGame` trước khi chạy cell clone.


In [ ]:
from pathlib import Path
import os

# Điền URL repo nếu muốn Colab tự clone. Ví dụ: "https://github.com/<owner>/<repo>.git"
# Nếu để trống, hãy upload/copy project vào PROJECT_DIR trước khi chạy bước clone.
REPO_URL = ""
REPO_REF = "main"  # có thể đổi thành tag, ví dụ "v0.1.4"
PROJECT_DIR = "/content/PianoGame"

# Khớp với GitHub Actions hiện tại của repo.
ANDROID_PLATFORM = "android-35"
ANDROID_BUILD_TOOLS = "35.0.0"
ANDROID_NDK_VERSION = "27.2.12479018"
ANDROID_HOME = "/content/android-sdk"

# Build cả ARM64 cho máy thật và x86_64 cho emulator như LDPlayer.
RUST_ANDROID_TARGETS = "aarch64-linux-android x86_64-linux-android"
TAURI_ANDROID_TARGETS = "aarch64 x86_64"

# Nếu chỉ muốn build nhanh để test máy thật, đổi thành:
# RUST_ANDROID_TARGETS = "aarch64-linux-android"
# TAURI_ANDROID_TARGETS = "aarch64"

SIGN_APK = True
KEYSTORE_PATH = "/content/pianogame-release.jks"
ANDROID_KEY_ALIAS = ""  # điền alias ở đây hoặc để trống để nhập ở bước ký APK

env_path = Path('/content/pianogame-colab-env.sh')
env_path.write_text(f'''\
export PROJECT_DIR="{PROJECT_DIR}"
export REPO_URL="{REPO_URL}"
export REPO_REF="{REPO_REF}"
export ANDROID_HOME="{ANDROID_HOME}"
export ANDROID_SDK_ROOT="{ANDROID_HOME}"
export ANDROID_NDK_HOME="{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}"
export ANDROID_NDK_ROOT="{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}"
export NDK_HOME="{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}"
export ANDROID_PLATFORM="{ANDROID_PLATFORM}"
export ANDROID_BUILD_TOOLS="{ANDROID_BUILD_TOOLS}"
export ANDROID_NDK_VERSION="{ANDROID_NDK_VERSION}"
export RUST_ANDROID_TARGETS="{RUST_ANDROID_TARGETS}"
export TAURI_ANDROID_TARGETS="{TAURI_ANDROID_TARGETS}"
export PATH="{ANDROID_HOME}/cmdline-tools/latest/bin:{ANDROID_HOME}/platform-tools:{ANDROID_HOME}/build-tools/{ANDROID_BUILD_TOOLS}:$HOME/.cargo/bin:$PATH"
''', encoding='utf-8')

os.environ.update({
    'PROJECT_DIR': PROJECT_DIR,
    'ANDROID_HOME': ANDROID_HOME,
    'ANDROID_SDK_ROOT': ANDROID_HOME,
    'ANDROID_NDK_HOME': f'{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}',
    'ANDROID_NDK_ROOT': f'{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}',
    'NDK_HOME': f'{ANDROID_HOME}/ndk/{ANDROID_NDK_VERSION}',
})

print(f'Đã ghi cấu hình môi trường vào {env_path}')
print(f'Project dir: {PROJECT_DIR}')


## 2. Cài Node.js 24 và Java 17


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh

apt-get update
apt-get install -y ca-certificates curl wget unzip zip git file build-essential openjdk-17-jdk

if ! command -v node >/dev/null 2>&1 || ! node --version | grep -Eq '^v24\.'; then
  curl -fsSL https://deb.nodesource.com/setup_24.x | bash -
  apt-get install -y nodejs
fi

node --version
npm --version
java -version


## 3. Cài Android SDK, Build Tools và NDK


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh

mkdir -p "$ANDROID_HOME/cmdline-tools"
cd /content

if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  rm -rf commandlinetools-linux-* cmdline-tools
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O commandlinetools-linux.zip
  unzip -q commandlinetools-linux.zip
  rm -rf "$ANDROID_HOME/cmdline-tools/latest"
  mv cmdline-tools "$ANDROID_HOME/cmdline-tools/latest"
fi

yes | sdkmanager --licenses >/dev/null || true
sdkmanager \
  "platform-tools" \
  "platforms;${ANDROID_PLATFORM}" \
  "build-tools;${ANDROID_BUILD_TOOLS}" \
  "ndk;${ANDROID_NDK_VERSION}"

sdkmanager --list_installed | grep -E 'platform-tools|platforms;|build-tools;|ndk;'


## 4. Cài Rust và Android Rust targets


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh

if [ ! -x "$HOME/.cargo/bin/rustup" ]; then
  curl https://sh.rustup.rs -sSf | sh -s -- -y
fi

source "$HOME/.cargo/env"
rustup default stable
for target in $RUST_ANDROID_TARGETS; do
  rustup target add "$target"
done

rustc --version
cargo --version
rustup target list --installed | grep -E 'android'


## 5. Lấy source code

Nếu `REPO_URL` trống, cell này sẽ dùng thư mục `/content/PianoGame` nếu bạn đã upload/copy source vào đó.


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh

if [ ! -d "$PROJECT_DIR" ]; then
  if [ -z "$REPO_URL" ]; then
    echo "PROJECT_DIR chưa tồn tại và REPO_URL đang trống. Hãy điền REPO_URL hoặc upload source vào $PROJECT_DIR." >&2
    exit 1
  fi
  git clone "$REPO_URL" "$PROJECT_DIR"
fi

cd "$PROJECT_DIR"
if [ -d .git ]; then
  git fetch --tags --force || true
  git checkout "$REPO_REF"
  git status --short
else
  echo "Dùng source đã upload trong $PROJECT_DIR vì thư mục này không phải git repo."
fi

pwd
ls -la | sed -n '1,80p'


## 6. Cài npm dependencies


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh
cd "$PROJECT_DIR"
npm ci


## 7. Kiểm tra frontend build

Bước này không bắt buộc vì Tauri sẽ chạy `beforeBuildCommand`, nhưng chạy riêng sẽ giúp bắt lỗi TypeScript/Vite sớm hơn.


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh
cd "$PROJECT_DIR"
npm run build


## 8. Khởi tạo Android project nếu chưa có, rồi patch icon và landscape

Cell này giữ cùng logic với GitHub Actions: copy icon từ `src-tauri/icons/android` vào generated Android resource và patch manifest để lock landscape.


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh
source "$HOME/.cargo/env"
cd "$PROJECT_DIR"

if [ ! -d src-tauri/gen/android ]; then
  npm run tauri:android:init
fi

manifest_path="src-tauri/gen/android/app/src/main/AndroidManifest.xml"
res_dir="src-tauri/gen/android/app/src/main/res"

if [ ! -f "$manifest_path" ]; then
  echo "AndroidManifest.xml không tồn tại tại $manifest_path" >&2
  exit 1
fi

mkdir -p "$res_dir"
cp -R src-tauri/icons/android/. "$res_dir/"

python3 <<'PY'
from pathlib import Path
import re

manifest = Path('src-tauri/gen/android/app/src/main/AndroidManifest.xml')
text = manifest.read_text(encoding='utf-8')

if re.search(r'android:icon="[^"]+"', text):
    text = re.sub(r'android:icon="[^"]+"', 'android:icon="@mipmap/ic_launcher"', text, count=1)
else:
    text = text.replace('<application', '<application\n        android:icon="@mipmap/ic_launcher"', 1)

if re.search(r'android:roundIcon="[^"]+"', text):
    text = re.sub(r'android:roundIcon="[^"]+"', 'android:roundIcon="@mipmap/ic_launcher_round"', text, count=1)
else:
    text = text.replace('<application', '<application\n        android:roundIcon="@mipmap/ic_launcher_round"', 1)

if re.search(r'android:screenOrientation="[^"]+"', text):
    text = re.sub(r'android:screenOrientation="[^"]+"', 'android:screenOrientation="landscape"', text, count=1)
else:
    text = text.replace('<activity', '<activity\n            android:screenOrientation="landscape"', 1)

manifest.write_text(text, encoding='utf-8')
PY

grep -E 'android:icon|android:roundIcon|android:screenOrientation' "$manifest_path"


## 9. Build APK

Mặc định build split APK cho `arm64-v8a` và `x86_64`. Nếu muốn nhanh hơn, quay lại bước cấu hình và chỉ để `aarch64`.


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh
source "$HOME/.cargo/env"
cd "$PROJECT_DIR"

npm run tauri -- android build --apk --target $TAURI_ANDROID_TARGETS --split-per-abi --ci


## 10. Gom APK unsigned


In [ ]:
%%bash
set -euo pipefail
source /content/pianogame-colab-env.sh
cd "$PROJECT_DIR"

rm -rf /content/pianogame-release-assets-unsigned
mkdir -p /content/pianogame-release-assets-unsigned

index=1
while IFS= read -r apk; do
  abi="apk-${index}"
  case "$apk" in
    *aarch64*|*arm64*) abi="arm64-v8a" ;;
    *x86_64*) abi="x86_64" ;;
  esac
  cp "$apk" "/content/pianogame-release-assets-unsigned/PianoGame-android-${abi}-release-unsigned.apk"
  index=$((index + 1))
done < <(find src-tauri/gen/android/app/build/outputs/apk -type f -name '*.apk' | grep release | grep -v debug | sort)

ls -lah /content/pianogame-release-assets-unsigned


## 11. Ký APK để cài được trên máy thật/LDPlayer

Nếu `SIGN_APK = True`, chạy cell này rồi upload file keystore `.jks` khi Colab hỏi. APK signed sẽ nằm trong `/content/pianogame-release-assets-signed`.


In [ ]:
import glob
import getpass
import os
import subprocess
from pathlib import Path

if not SIGN_APK:
    print('SIGN_APK = False, bỏ qua bước ký APK.')
else:
    keystore = Path(KEYSTORE_PATH)
    if not keystore.exists():
        print(f'Chưa thấy keystore tại {KEYSTORE_PATH}. Hãy upload file .jks/.keystore.')
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('Chưa upload keystore.')
        first_name = next(iter(uploaded.keys()))
        uploaded_path = Path('/content') / first_name
        uploaded_path.rename(keystore)
        print(f'Đã lưu keystore vào {keystore}')

    alias = ANDROID_KEY_ALIAS or input('Android key alias: ').strip()
    if not alias:
        raise RuntimeError('Alias không được để trống.')

    keystore_password = getpass.getpass('Keystore password: ')
    key_password = getpass.getpass('Key password: ')

    unsigned_dir = Path('/content/pianogame-release-assets-unsigned')
    signed_dir = Path('/content/pianogame-release-assets-signed')
    signed_dir.mkdir(parents=True, exist_ok=True)

    unsigned_apks = sorted(unsigned_dir.glob('*.apk'))
    if not unsigned_apks:
        raise RuntimeError(f'Không tìm thấy APK unsigned trong {unsigned_dir}')

    android_home = os.environ.get('ANDROID_HOME', ANDROID_HOME)
    zipalign = Path(android_home) / 'build-tools' / ANDROID_BUILD_TOOLS / 'zipalign'
    apksigner = Path(android_home) / 'build-tools' / ANDROID_BUILD_TOOLS / 'apksigner'

    for tool in (zipalign, apksigner):
        if not tool.exists():
            raise RuntimeError(f'Không tìm thấy tool: {tool}')

    for apk in unsigned_apks:
        aligned = Path('/content') / f'{apk.stem}-aligned.apk'
        signed = signed_dir / apk.name.replace('-unsigned.apk', '-signed.apk')

        subprocess.run([str(zipalign), '-p', '-f', '4', str(apk), str(aligned)], check=True)
        subprocess.run([
            str(apksigner), 'sign',
            '--ks', str(keystore),
            '--ks-key-alias', alias,
            '--ks-pass', f'pass:{keystore_password}',
            '--key-pass', f'pass:{key_password}',
            '--out', str(signed),
            str(aligned),
        ], check=True)
        subprocess.run([str(apksigner), 'verify', '--verbose', str(signed)], check=True)
        print(f'Đã ký: {signed}')

    print('\nAPK signed:')
    for path in sorted(signed_dir.glob('*.apk')):
        print(path, path.stat().st_size, 'bytes')


## 12. Tải APK về máy


In [ ]:
from pathlib import Path
from google.colab import files

signed_dir = Path('/content/pianogame-release-assets-signed')
unsigned_dir = Path('/content/pianogame-release-assets-unsigned')
download_dir = signed_dir if signed_dir.exists() and any(signed_dir.glob('*.apk')) else unsigned_dir

apks = sorted(download_dir.glob('*.apk'))
if not apks:
    raise RuntimeError('Không tìm thấy APK để tải.')

for apk in apks:
    print(f'Tải: {apk}')
    files.download(str(apk))


## Gợi ý tăng tốc

- Nếu chỉ test máy thật Android ARM64, ở bước cấu hình chỉ build `aarch64` thay vì cả `aarch64 x86_64`.
- Giữ nguyên Colab runtime sau lần build đầu để tận dụng cache npm/Cargo/Gradle trong runtime.
- Nếu cần cache qua nhiều runtime Colab khác nhau, có thể mount Google Drive rồi trỏ cache Gradle/Cargo/npm sang Drive, nhưng lần đầu đồng bộ Drive đôi khi cũng chậm.
- Khi chỉ chỉnh frontend/CSS, build lại trong cùng runtime thường nhanh hơn đáng kể vì Android SDK, NDK, Rust target và Gradle dependency đã có sẵn.
